# GovConnect Edge — Whisper-Small LoRA Fine-Tuning on IndicVoices (Colab T4 GPU)

This notebook trains a dialect-adaptive Whisper-Small speech recognition model using PEFT LoRA on the AI4Bharat IndicVoices dataset (Tamil & Hindi), then converts it to CTranslate2 INT8 format for deployment on GovConnect Edge.

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Install Dependencies
!pip install -q transformers datasets evaluate jiwer accelerate peft bitsandbytes librosa soundfile ctranslate2

In [ ]:
# 3. Hugging Face Login (Optional if private repo)
from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# 4. Configuration
LANG = "ta"  # 'ta' (Tamil) or 'hi' (Hindi)
MODEL_ID = "openai/whisper-small"
OUTPUT_DIR = f"./whisper_small_lora_{LANG}"
BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 3
LR = 1e-3

In [ ]:
# 5. Load Processor & 8-Bit Model
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANG, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, load_in_8bit=True, device_map="auto")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 6. Load IndicVoices Subset
from datasets import load_dataset, Audio

print(f"Downloading IndicVoices for {LANG}...")
dataset = load_dataset("ai4bharat/indicvoices_r", LANG, trust_remote_code=True)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
print(dataset)

In [ ]:
# 7. Train with Seq2SeqTrainer
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

def prepare(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

train_ds = dataset["train"].select(range(min(2000, len(dataset["train"])))).map(prepare, remove_columns=dataset["train"].column_names, num_proc=2)
eval_ds = dataset["validation"].select(range(min(200, len(dataset["validation"])))).map(prepare, remove_columns=dataset["validation"].column_names, num_proc=2)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=50,
    num_train_epochs=EPOCHS,
    evaluation_strategy="epoch",
    fp16=True,
    save_strategy="epoch",
    logging_steps=25,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    tokenizer=processor.feature_extractor,
)

trainer.train()

In [ ]:
# 8. Save Weights and Export to CTranslate2 INT8
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

# Merge LoRA & export to CTranslate2 INT8
!ct2-transformers-converter --model {OUTPUT_DIR} --output_dir {OUTPUT_DIR}_ct2_int8 --quantization int8 --low_cpu_mem_usage --force
print("Finished export to CTranslate2 INT8!")